In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df=pd.read_csv("Data/loan_data_without_outliers.csv")
df

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38248,24.0,female,Associate,31924.0,2,RENT,12229.0,MEDICAL,10.70,0.38,4.0,678,No,1
38249,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
38250,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
38251,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


In [3]:
#doing a train test split in the ratio of 70:30
x_train,x_test,y_train,y_test=train_test_split(df.iloc[:,0:13],df.loan_status,test_size=.30,random_state=1)

## 1) Data-preprocessing 

In [ ]:
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [6]:
# Create a ColumnTransformer to apply different preprocessing steps to different column groups
transformer=ColumnTransformer(transformers=[
    # Encode education level as ordinal values (ordered categories Low to High)
    ("person_education_encoding",OrdinalEncoder(categories=[['High School','Associate','Bachelor','Master','Doctorate']]),["person_education"]),
    
    # Encode loan default history as ordinal values (No=0, Yes=1)
    ("previous_loan_defaults_on_file_encoding",OrdinalEncoder(categories=[["No","Yes"]]),["previous_loan_defaults_on_file"]),
    
    # Convert categorical features to binary columns using one-hot encoding
    ("one_hot_encoding",OneHotEncoder(),["person_gender","person_home_ownership","loan_intent"]),
    
],remainder="passthrough")
# Fit the transformer on training data and apply transformations
x_train=transformer.fit_transform(x_train)
# Apply same transformations to test data (without refitting)
x_test=transformer.transform(x_test)

## 2) Random forest and model evaluation 

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

In [ ]:
#traning a model with default hyperparameters 
model=RandomForestClassifier(random_state=1)

model.fit(x_train,y_train)

y_test_predict=model.predict(x_test)

print(classification_report(y_test,y_test_predict))

              precision    recall  f1-score   support

           0       0.94      0.97      0.95      8841
           1       0.89      0.77      0.83      2635

    accuracy                           0.93     11476
   macro avg       0.91      0.87      0.89     11476
weighted avg       0.92      0.93      0.92     11476



From the above classification report we see that without changing any hyperparameters  we have accuracy of 93% and F1 score of 83% in the case of yes(1) for loan approval